# Quantum-Inspired Optimization for Construction Crew Scheduling

This notebook asks a practical question: **can an energy-based optimizer produce a shorter construction schedule than a fast classical rule when crews are limited?** It builds the schedule as a QUBO, solves it with classical simulated annealing, and compares it with a CPM-priority list scheduler. A final micro-scale experiment runs QAOA on a local simulator.

> **Terminology note.** CPM alone ignores crew limits and is therefore a lower bound, not necessarily a feasible schedule. D-Wave simulated annealing is a thermal classical heuristic operating on a QUBO/Ising energy landscape; it does not reproduce quantum tunnelling. QAOA is a quantum algorithm, but here it is simulated classically.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import platform

import dimod
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from crew_schedule import (build_qubo, construction_project, micro_projects,
    run_benchmarks, solve_annealing, solve_cpm_list, solve_qaoa, toy_project,
    tune_annealing, validate_schedule)
from crew_schedule.experiments import consistency_summary
from crew_schedule.visualization import (plot_consistency, plot_dependency_graph,
    plot_gantt_comparison, plot_qaoa_scaling, plot_qubo_matrix, plot_scaling,
    plot_tuning, save_figure)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGURES = ROOT / 'figures'
RESULTS = ROOT / 'results'
FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 30)
print(f'Python {platform.python_version()} | project root: {ROOT}')

## 1. The construction project

The main example has ten non-preemptive tasks and three interchangeable crews. Each task consumes one crew for its full integer duration. An arrow means the predecessor must finish before the successor starts.

In [ ]:
project = construction_project()
task_table = pd.DataFrame([{'id': t.id, 'task': t.name, 'duration': t.duration} for t in project.tasks])
display(task_table)
fig, _ = plot_dependency_graph(project)
save_figure(fig, FIGURES / 'construction_dependencies.png')
plt.show()

## 2. From schedule to QUBO

For task $i$ and allowed start slot $t$, let $x_{i,t}=1$ mean that the task starts then. We add a zero-duration finish milestone $F$ after all terminal tasks and minimize

$$\min \sum_t t x_{F,t}.$$

Three penalty families turn the constrained problem into one unconstrained quadratic energy:

1. **Start once:** $A(\sum_t x_{i,t}-1)^2$.
2. **Respect precedence:** add $A x_{i,t}x_{j,s}$ whenever $s<t+d_i$.
3. **Respect crews:** at each slot, use binary slack to penalize active tasks above the crew limit.

We choose $A=H+1$. Since the finish objective ranges only from 0 to $H$, one unit of violation costs more than any possible objective improvement. Earliest/latest precedence bounds remove impossible start variables before the BQM is built.

In [ ]:
toy = toy_project()
toy_bqm, toy_encoding = build_qubo(toy)
toy_exact = dimod.ExactSolver().sample(toy_bqm)
from crew_schedule.qubo import decode_sample
toy_schedule, toy_errors = decode_sample(toy_exact.first.sample, toy_encoding)
toy_summary = pd.DataFrame([{
    'real tasks': len(toy.tasks), 'binary variables': toy_bqm.num_variables,
    'interactions': toy_bqm.num_interactions, 'ground energy': toy_exact.first.energy,
    'decoded makespan': toy_schedule.makespan, 'valid': not toy_errors}])
display(toy_summary)
assert toy_schedule.makespan == 4 and validate_schedule(toy, toy_schedule)[0]
fig, _ = plot_qubo_matrix(toy_bqm)
save_figure(fig, FIGURES / 'toy_qubo_matrix.png')
plt.show()

## 3. Classical baseline and annealing tuning

The classical algorithm gives ready tasks priority according to their remaining critical-path length. Unlike raw CPM, the resulting list schedule never starts more work than the available crews can support. Annealing is stochastic, so four parameter settings are compared across three fixed seeds.

In [ ]:
classical = solve_cpm_list(project)
best_config, tuning = tune_annealing(project)
tuning.to_csv(RESULTS / 'annealing_tuning.csv', index=False)
print('Selected configuration:', best_config)
display(tuning.groupby('config', sort=False).agg(
    feasible_run_rate=('feasible', 'mean'),
    median_makespan=('makespan', 'median'),
    median_runtime=('runtime', 'median')).round(4))
fig, _ = plot_tuning(tuning)
save_figure(fig, FIGURES / 'annealing_tuning.png')
plt.show()

In [ ]:
best_params = asdict(best_config)
best_params.pop('name')
main_runs = [solve_annealing(project, seed=seed, **best_params) for seed in (707, 808, 909)]
feasible_main = [run for run in main_runs if run.feasible]
assert feasible_main, 'No feasible annealing schedule was found for the main example.'
annealed = min(feasible_main, key=lambda run: (run.schedule.makespan, run.energy))
comparison = pd.DataFrame([
    {'method': classical.method, 'makespan': classical.schedule.makespan,
     'runtime_s': classical.runtime, 'feasible': classical.feasible,
     'lower_bound': classical.metadata['combined_lower_bound']},
    {'method': annealed.method, 'makespan': annealed.schedule.makespan,
     'runtime_s': annealed.runtime, 'feasible': annealed.feasible,
     'lower_bound': classical.metadata['combined_lower_bound']}])
display(comparison.round(5))
comparison.to_csv(RESULTS / 'main_comparison.csv', index=False)
assert classical.feasible and annealed.feasible
fig, _ = plot_gantt_comparison(project, {
    'CPM-priority list': classical.schedule, 'Simulated annealing': annealed.schedule})
save_figure(fig, FIGURES / 'construction_gantt_comparison.png')
plt.show()

## 4. Scaling across randomized projects

The reproducible benchmark covers 4, 6, 8, 10, and 12 tasks; two and three crews; two generated DAGs in each cell; and three independent annealing seeds. Medians and interquartile ranges are more informative than a single lucky stochastic run.

In [ ]:
benchmark = run_benchmarks(annealing_params=best_params)
benchmark.to_csv(RESULTS / 'random_benchmarks.csv', index=False)
consistency = consistency_summary(benchmark)
consistency.to_csv(RESULTS / 'consistency_summary.csv', index=False)
display(benchmark.groupby(['task_count', 'crew_limit']).agg(
    classical_makespan=('classical_makespan', 'median'),
    annealing_makespan=('annealing_makespan', 'median'),
    feasible_run_rate=('annealing_feasible', 'mean'),
    median_gap=('gap_to_lower_bound', 'median'),
    annealing_runtime_s=('annealing_runtime', 'median')).round(4))
assert benchmark.loc[benchmark['annealing_feasible'], 'annealing_makespan'].notna().all()

In [ ]:
fig, _ = plot_scaling(benchmark)
save_figure(fig, FIGURES / 'scaling_quality_runtime.png')
plt.show()
fig, _ = plot_consistency(benchmark)
save_figure(fig, FIGURES / 'annealing_consistency.png')
plt.show()

## 5. QAOA: a quantum algorithm on a classical simulator

Every BQM variable becomes a qubit. State-vector memory therefore grows exponentially, while the time-indexed model's variable count grows with both tasks and horizon. QAOA is run only for the 2–4 task micro-projects, using depths $p=1,2$, three seeds, 1,024 shots, COBYLA with at most 100 evaluations, and an 18-qubit cap. The ten-task model is explicitly reported as not run. Exact NumPy eigensolver energies verify each micro BQM.

In [ ]:
qaoa_rows = []
for micro in micro_projects():
    for reps in (1, 2):
        for seed in (11, 22, 33):
            result = solve_qaoa(micro, reps=reps, seed=seed, shots=1024, maxiter=100)
            qaoa_rows.append({
                'instance': micro.name, 'task_count': len(micro.tasks), 'reps': reps,
                'seed': seed, 'status': result.metadata['status'],
                'feasible': result.feasible,
                'makespan': result.schedule.makespan if result.schedule else np.nan,
                'energy': result.energy, 'exact_energy': result.metadata.get('exact_ground_energy'),
                'qubits': result.metadata['qubits'], 'runtime': result.runtime,
                'feasible_probability': result.metadata.get('feasible_probability')})
large_qaoa = solve_qaoa(project, reps=1, seed=11)
qaoa_rows.append({'instance': project.name, 'task_count': len(project.tasks), 'reps': 1,
    'seed': 11, 'status': large_qaoa.metadata['status'], 'feasible': False,
    'makespan': np.nan, 'energy': np.nan, 'exact_energy': np.nan,
    'qubits': large_qaoa.metadata['qubits'], 'runtime': large_qaoa.runtime,
    'feasible_probability': np.nan})
qaoa_results = pd.DataFrame(qaoa_rows)
qaoa_results.to_csv(RESULTS / 'qaoa_micro_results.csv', index=False)
display(qaoa_results.groupby(['task_count', 'reps', 'status'], dropna=False).agg(
    feasible_run_rate=('feasible', 'mean'), median_makespan=('makespan', 'median'),
    median_runtime_s=('runtime', 'median'), qubits=('qubits', 'max')).round(4))
assert set(qaoa_results[qaoa_results['task_count'] <= 4]['status']) == {'completed'}
fig, _ = plot_qaoa_scaling(qaoa_results)
save_figure(fig, FIGURES / 'qaoa_scaling.png')
plt.show()

## 6. Conclusions and limits

- The CPM/resource lower bound tells us what no feasible schedule can beat; the CPM-priority list algorithm supplies a very fast feasible baseline.
- Simulated annealing can match or improve a heuristic baseline, but quality and feasibility vary with random seed and annealing settings. Runtime must therefore be reported with consistency, not from one run.
- QAOA uses the same QUBO but is practical here only for micro-instances. The local experiment demonstrates the gate-based workflow; it is not evidence of quantum speedup.
- Results apply to the simplified assumptions: interchangeable crews, one crew per task, integer durations, no preemption, and no calendars or skill types. A production RCPSP would require richer resources and a stronger exact benchmark.

In [ ]:
summary = {
    'main_lower_bound': int(classical.metadata['combined_lower_bound']),
    'classical_makespan': int(classical.schedule.makespan),
    'annealing_makespan': int(annealed.schedule.makespan),
    'benchmark_feasible_run_rate': float(benchmark['annealing_feasible'].mean()),
    'qaoa_micro_feasible_run_rate': float(qaoa_results[qaoa_results['task_count'] <= 4]['feasible'].mean()),
}
pd.Series(summary, name='value').to_csv(RESULTS / 'executive_summary.csv')
display(pd.Series(summary, name='value'))
print(f'Wrote {len(list(FIGURES.glob("*.png")))} figures and {len(list(RESULTS.glob("*.csv")))} result tables.')